# SFT 对话数据 → .pt 分片文件（6 步清洗 + tokenize）
#
# ★ Cell 3：综合清洗 — 6 步子串匹配（参考 clean_base_pretrain_data.ipynb）
#   ① AI 身份声明  ② 代码/标记噪声  ③ 纯英文(无数学)
#   ④ 非中英文语种  ⑤ 中外文翻译  ⑥ 文本规范化
# ★ 后续 cell：格式化聊天 + 分词 → [N, block_size] tokens + loss_mask
# ★ mask 策略：只计算 assistant 回复的 loss，user/system=0.0，1→0 过渡额外 2 token
# ★ 角色标签：user：/ assistant：/ system：
#
# 文件链：raw → cleaned（Cell 3 综合清洗，一步到位）→ tokenize（Cell 7-8）

In [ ]:
import json
import os
import time
from pathlib import Path

import torch
from tokenizers import Tokenizer

# 自动适配本地/DSW 路径
BASE = Path.cwd()
# 如果 tokenizer 不在当前目录，尝试 shayler2.0 子目录
if not (BASE / "tokenizer_minimind_8k").exists() and (BASE / "shayler2.0").exists():
    BASE = BASE / "shayler2.0"
print(f"项目根目录: {BASE}")
print(f"tokenizer: {'✅' if (BASE / 'tokenizer_minimind_8k').exists() else '❌'}")

In [ ]:
# ============ CONFIG ============
TOKENIZER_PATH = BASE / "tokenizer_minimind_8k" / "tokenizer.json"
# ★ 自动选择最佳可用输入：cleaned > merged > v7 > raw
INPUT_PATH = BASE / "data_stage1" / "sft_t2t.jsonl"  # 兜底
for _candidate in ["sft_t2t_cleaned.jsonl", "sft_stage1_stage2_merged.jsonl",
                    "sft_t2t_cleaned_v7.jsonl", "sft_t2t.jsonl"]:
    _p = BASE / "data_stage1" / _candidate
    if _p.exists():
        INPUT_PATH = _p
        break
OUTPUT_DIR = BASE / "data_stage1" / "sft_pt"
BLOCK_SIZE = 1024
SAMPLES_PER_FILE = 20000

ROLE_TAGS = {
    "user": "user：", "assistant": "assistant：", "system": "system：",
    "human": "user：", "gpt": "assistant：", "bot": "assistant：", "tool": "system：",
}

# 加载分词器
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print(f"分词器: {TOKENIZER_PATH} {'✅' if TOKENIZER_PATH.exists() else '❌ 不存在'}")
print(f"输入:   {INPUT_PATH} {'✅' if INPUT_PATH.exists() else '❌ 不存在'}")
print(f"输出:   {OUTPUT_DIR}")

In [ ]:
# ===== 6 步清洗：AI 身份直接匹配 + 其他原逻辑 =====
import json, time, re as _re
from collections import defaultdict

CLEANED_PATH = BASE / "data_stage1" / "sft_t2t_cleaned.jsonl"
RAW_PATH = BASE / "data_stage1" / "sft_t2t.jsonl"
for _src in ["sft_t2t.jsonl", "sft_stage1_stage2_merged.jsonl",
             "sft_t2t_cleaned_v7.jsonl"]:
    _p = BASE / "data_stage1" / _src
    if _p.exists():
        RAW_PATH = _p
        break

# ═══ ① AI 身份关键词 — 直接子串匹配删除 ═══
IDENTITY_KW = [
    "通义千问", "Qwen", "文心一言", "ERNIE", "讯飞星火",
    "ChatGPT", "OpenAI", "GPT-4", "GPT-4o", "GPT-3.5", "GPT-3",
    "Claude", "Anthropic", "Kimi", "月之暗面",
    "DeepSeek", "深度求索", "Gemini",
    "百川智能", "Baichuan", "ChatGLM", "智谱清言", "MOSS",
    "SenseChat", "360智脑", "Llama", "Mistral", "Mixtral",
    "紫东太初", "书生浦语", "悟道", "盘古大模型",
    "MiniMax", "Yi-Large", "Yi-Chat",
    "腾讯混元", "腾讯元宝", "豆包", "天工AI",
    "我是AI", "我是一个AI", "我是个AI", "我是人工智能",
    "我是语言模型", "我是大语言模型", "我是一个语言模型",
    "我是大模型", "我是一个大模型", "我是大型语言模型", "我是一个大型语言模型",
    "我是生成式AI", "我是生成式人工智能",
    "我是虚拟助手", "我是智能助手", "我是AI助手",
    "我是一个AI助手", "我是人工智能助手",
    "我是聊天机器人", "我是AI聊天机器人",
    "我是对话系统", "我是AI对话系统",
    "我是文本生成模型", "我是一个聊天程序", "我是AI程序", "我是一个AI程序",
    "我是你的AI", "我是你的智能", "我是你的AI助手", "我是你的智能助手",
    "作为AI", "作为一个AI", "作为一个人工智能", "作为人工智能",
    "身为AI", "身为一个人工智能", "身为人工智能",
    "因为我是AI", "由于我是AI", "因为我是人工智能",
    "因为我是语言模型", "由于我是语言模型",
    "作为语言模型", "作为一个语言模型", "身为语言模型",
    "作为大语言模型", "作为一个大语言模型",
    "我只是AI", "我只是一个AI", "我只是人工智能",
    "我只是个AI", "我只是个程序", "我只是一个程序", "我只是一个模型",
    "我仅仅是一个AI", "我仅仅是个AI", "我只是个语言模型",
    "我不是人类", "我不是真正的人", "我不是真实的人",
    "我并不是真正的人", "我并不具有人类",
    "我没有人类的", "我不具备人类",
    "我没有感情", "我没有情感", "我没有情绪",
    "我没有感受", "我没有意识", "我没有自我意识",
    "我没有实体", "我没有身体", "我没有物理形态",
    "我没有真实情感", "我没有人类情感",
    "我没有人类的感情", "我没有人的情感",
    "没有真实的感情", "没有真实的情感",
    "我不能像人类", "我无法像人类",
    "我不能体验", "我无法体验", "我无法感受",
    "我不能感受", "我不能理解情感", "我不能产生情感",
    "我是一个被训练的", "我是一个被开发的",
    "我是被训练出来的", "我是被开发出来的",
    "我被设计用来帮助", "我被训练来帮助",
    "作为AI助手", "作为人工智能助手",
    "我的创造者", "我的开发者", "我的设计者是",
    "由人工智能技术驱动",
    "我的知识截止于", "我的训练数据截止", "我的知识更新时间",
    "我是一个由", "由深度神经网络构成的",
    "很高兴为你服务", "有什么我可以帮你的", "有什么我可以帮助你的",
    "jingyaogong", "minimind",
]

def has_ai_identity(text):
    tl = text.lower()
    for kw in IDENTITY_KW:
        if kw.lower() in tl:
            return True
    return False

# ═══ ② 代码/标记噪声 ═══
CODE_MARKERS = [
    "\\begin{", "\\end{", "\\frac", "\\sqrt", "\\sum", "\\int",
    "\\alpha", "\\beta", "\\gamma", "\\delta", "\\lambda",
    "\\mathbb", "\\mathcal", "\\mathbf", "\\text",
    "\\left", "\\right", "\\cdot", "\\times", "\\infty",
    "|---", "| ---", "|:---", "|---|", "| :---",
    "```", "<!--", "-->", "<?xml", "<!DOCTYPE", "<html", "</html",
    "&nbsp;", "&lt;", "&gt;", "&amp;", "&quot;", "&#",
    "import ", "def ", "class ", "function ",
    "const ", "let ", "var ", "print(", "return ",
    "===", "***", "----",
]

def is_code_noise(text):
    if sum(1 for m in CODE_MARKERS if m in text) >= 2: return True
    if text.count("$") >= 4: return True
    if text.count("|") >= 5: return True
    if len(text) > 50 and text.count("\n") / len(text) > 0.08: return True
    return False

# ═══ ③ 英文检测（无数学→移除）═══
def _cjk_ratio(text):
    if not text: return 0.0
    cjk = sum(1 for c in text if (
        '一' <= c <= '鿿' or '㐀' <= c <= '䶿' or
        '豈' <= c <= '﫿' or '　' <= c <= '〿' or '＀' <= c <= '￯'))
    return cjk / len(text)

def _has_math(text):
    tl = text.lower()
    if text.count('$') >= 2: return True
    if any(c in text for c in ['\\frac','\\sqrt','\\sum','\\int','\\cdot',
        '\\times','\\pi','\\theta','\\alpha','\\beta','\\gamma','\\lambda']):
        return True
    mk = ['equation','solve','solving','formula','theorem','calculate','derivative',
        'integral','polynomial','matrix','vector','algebra','geometry','trigonometry',
        'calculus','fraction','decimal','exponent','logarithm','quadratic',
        'probability','statistics','graph of','slope of','perpendicular',
        'right triangle','pythagorean','circumference','diameter','radius',
        'arithmetic','find the value','what is the sum','evaluate the',
        'simplify the','compute the','solve for x','solve for y','find x','find y']
    if sum(1 for kw in mk if kw in tl) >= 1: return True
    if len(_re.findall(r'\d[\d\s.xXyYzZ]*[+\-*/×÷][\d\s.xXyYzZ]*=', text)) >= 2: return True
    if len(text) > 0 and sum(1 for c in text if c in '=+-*/^') / len(text) > 0.03: return True
    return False

def is_non_math_english(text):
    if len(text) < 100: return False
    if _cjk_ratio(text) >= 0.02: return False
    if _has_math(text): return False
    return True

# ═══ ④ 非中英文语种 ═══
def is_foreign_language(text):
    for c in text:
        cp = ord(c)
        if 0x3040 <= cp <= 0x309F: return True
        if 0x30A0 <= cp <= 0x30FF: return True
        if 0xAC00 <= cp <= 0xD7AF: return True
        if 0x0400 <= cp <= 0x04FF: return True
        if 0x0600 <= cp <= 0x06FF: return True
        if 0x0E00 <= cp <= 0x0E7F: return True
    foreign = 0
    for c in text:
        cp = ord(c)
        if ('一' <= c <= '鿿' or '㐀' <= c <= '䶿' or
            '豈' <= c <= '﫿' or '　' <= c <= '〿' or '＀' <= c <= '￯'):
            continue
        if cp < 128: continue
        if 0x00C0 <= cp <= 0x00FF: foreign += 1; continue
        if cp >= 128: foreign += 1
    return foreign / max(len(text), 1) > 0.05

# ═══ ⑤ 中外文翻译（文言文保留）═══
_TRANS_TRIG = ["翻译成","翻译为","翻译下列","翻译以下","请翻译",
    "把下列","把以下","将下列","将以下","译成","译为",
    "请将下列","请将以下","请把下列","请把以下",
    "translate the following","translate this","translate into","translation:",
    "原文：","译文：","原文:","译文:","英文原文","中文译文","参考译文"]
_FOREIGN_LANG = ["英文","英语","英文版","英译","日文","日语","日译",
    "韩文","韩语","韩译","法文","法语","法译","德文","德语","德译",
    "俄文","俄语","俄译","西班牙语","西班牙文","阿拉伯语","阿拉伯文",
    "English","Japanese","Korean","French","German",
    "Russian","Spanish","Arabic","in English","in Japanese","in French"]
_CLASSICAL = ["文言文","古文","古诗","古诗词","古汉语","文言","唐诗","宋词",
    "元曲","诗经","论语","史记","世说新语","古文观止","现代汉语","现代文",
    "白话文","白话","用现代","译为现代"]

def is_translation_data(text):
    tl = text.lower()
    if any(m in text for m in _CLASSICAL): return False
    if not any(kw in tl for kw in _TRANS_TRIG): return False
    if any(kw in tl for kw in _FOREIGN_LANG): return True
    tags = ["英文：","中文：","英文:","中文:","English：","Chinese："]
    if sum(1 for t in tags if t in text) >= 2: return True
    return False

# ═══ ⑥ 文本规范化 ═══
def normalize_text(text):
    text = text.replace(" ", " ")
    text = text.replace("&nbsp;", " ")
    text = text.replace("&amp;", "&")
    text = text.replace("&lt;", "<")
    text = text.replace("&gt;", ">")
    text = text.replace("&quot;", "\"")
    while "\n\n\n" in text: text = text.replace("\n\n\n", "\n\n")
    lines = []
    for line in text.split("\n"):
        while "  " in line: line = line.replace("  ", " ")
        lines.append(line.strip())
    return "\n".join(lines).strip()

# ═══════════════════════════════════════
if CLEANED_PATH.exists():
    print(f"已存在: {CLEANED_PATH.name} ({CLEANED_PATH.stat().st_size / 1e6:.1f} MB)")
    print(f"如需重新清洗请先删除此文件")
else:
    print(f"开始清洗: {RAW_PATH.name}")
    print(f"大小: {RAW_PATH.stat().st_size / 1e9:.2f} GB\n")

    total = r_identity = r_code = r_english = r_foreign = r_trans = r_other = 0
    normalized = 0
    removed_lines, cat_counter = [], defaultdict(int)
    t0 = time.time()

    fin = open(RAW_PATH, "r", encoding="utf-8")
    fout = open(CLEANED_PATH, "w", encoding="utf-8")
    for line in fin:
        ls = line.strip()
        if not ls: continue
        total += 1
        try:
            obj = json.loads(ls)
        except json.JSONDecodeError:
            r_other += 1; continue
        msgs = obj.get("conversations") or obj.get("messages") or []
        if len(msgs) < 2:
            r_other += 1; continue
        text = " ".join(m.get("content", "") for m in msgs)
        if len(text) < 10:
            r_other += 1; continue

        # ── ① AI 身份 — 直接匹配删除 ──
        if has_ai_identity(text):
            r_identity += 1; cat_counter["AI身份"] += 1
            removed_lines.append({"line_no": total, "category": "AI身份", "text": text[:300]})
            continue

        # ── ② 代码噪声 ──
        if is_code_noise(text):
            r_code += 1; cat_counter["代码噪声"] += 1
            continue

        # ── ③ 纯英文(无数学) ──
        if is_non_math_english(text):
            r_english += 1; cat_counter["纯英文"] += 1
            continue

        # ── ④ 非中英文语种 ──
        if is_foreign_language(text):
            r_foreign += 1; cat_counter["非中英文语种"] += 1
            continue

        # ── ⑤ 中外文翻译 ──
        if is_translation_data(text):
            r_trans += 1; cat_counter["中外文翻译"] += 1
            continue

        # ── ⑥ 规范化 + 写入 ──
        changed = False
        for m in msgs:
            if "content" in m:
                ct = normalize_text(m["content"])
                if ct != m["content"]:
                    m["content"] = ct
                    changed = True
        if changed: normalized += 1
        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

        if total % 500000 == 0:
            e = time.time() - t0
            print(f"  {total/1e6:.1f}M | AI身份 {r_identity:,} | 代码 {r_code:,} | "
                  f"英文 {r_english:,} | 外语 {r_foreign:,} | 翻译 {r_trans:,} | {e:.0f}s")

    fin.close()
    fout.close()

    elapsed = time.time() - t0
    removed_total = r_identity + r_code + r_english + r_foreign + r_trans + r_other
    kept = total - removed_total

    print(f"\n{'='*50}")
    print(f"清洗完成 - {elapsed:.0f}s")
    print(f"{'='*50}")
    print(f"  总行数:         {total:,}")
    print(f"  ① AI 身份:     {r_identity:>8,} ({r_identity/total*100:.1f}%)")
    print(f"  ② 代码噪声:    {r_code:>8,} ({r_code/total*100:.1f}%)")
    print(f"  ③ 纯英文:      {r_english:>8,} ({r_english/total*100:.1f}%)")
    print(f"  ④ 非中英文语种:{r_foreign:>8,} ({r_foreign/total*100:.1f}%)")
    print(f"  ⑤ 中外文翻译:  {r_trans:>8,} ({r_trans/total*100:.1f}%)")
    print(f"  ⑥ 其他:        {r_other:>8,}")
    print(f"  文本规范化:    {normalized:>8,}")
    print(f"  保留:          {kept:>8,} ({kept/total*100:.1f}%)")

    for cat in sorted(cat_counter.keys(), key=lambda c: cat_counter[c], reverse=True):
        print(f"  {cat}: {cat_counter[cat]:,}")
    in_mb = RAW_PATH.stat().st_size / 1e6
    out_mb = CLEANED_PATH.stat().st_size / 1e6
    print(f"\n文件: {in_mb:.0f} MB -> {out_mb:.0f} MB")
    print(f"DONE: {CLEANED_PATH.name}")


In [ ]:
# ===== Cell 3.5：逐行文本匹配，彻底删除 AI 身份残留 =====
# 不做 JSON 解析，直接对整行原始文本做子串匹配，命中即删

SRC = BASE / "data_stage1" / "sft_t2t_cleaned.jsonl"
DST = BASE / "data_stage1" / "sft_t2t_cleaned_v8.jsonl"

KW = [
    '通义千问','Qwen','qwen','文心一言','ERNIE','讯飞星火',
    'ChatGPT','chatgpt','OpenAI','GPT-4','GPT-4o','GPT-3','GPT-3.5',
    'Claude','claude','Anthropic','Kimi','kimi','月之暗面',
    'DeepSeek','deepseek','深度求索','Gemini','gemini',
    '百川智能','Baichuan','ChatGLM','智谱清言','MOSS','moss',
    'SenseChat','360智脑','Llama','llama','Mistral','Mixtral',
    '紫东太初','书生浦语','悟道','盘古大模型','MiniMax','Yi-Large','Yi-Chat',
    '腾讯混元','腾讯元宝','豆包','天工AI',
    '我是AI','我是一个AI','我是个AI','我是人工智能','我是一个人工智能',
    '我是语言模型','我是大语言模型','我是一个语言模型',
    '我是大模型','我是一个大模型','我是大型语言模型','我是一个大型语言模型',
    '我是生成式AI','我是生成式人工智能',
    '我是虚拟助手','我是智能助手','我是AI助手',
    '我是一个AI助手','我是人工智能助手',
    '我是聊天机器人','我是AI聊天机器人',
    '我是对话系统','我是AI对话系统',
    '我是文本生成模型','我是一个聊天程序','我是AI程序','我是一个AI程序',
    '我是你的AI','我是你的智能','我是你的AI助手','我是你的智能助手',
    '作为AI','作为一个AI','作为一个人工智能','作为人工智能',
    '身为AI','身为一个人工智能','身为人工智能',
    '因为我是AI','由于我是AI','因为我是人工智能',
    '因为我是语言模型','由于我是语言模型',
    '作为语言模型','作为一个语言模型','身为语言模型',
    '作为大语言模型','作为一个大语言模型',
    '我只是AI','我只是一个AI','我只是人工智能',
    '我只是个AI','我只是个程序','我只是一个程序','我只是一个模型',
    '我仅仅是一个AI','我仅仅是个AI','我只是个语言模型',
    '我不是人类','我不是真正的人','我不是真实的人',
    '我并不是真正的人','我并不具有人类',
    '我没有人类的','我不具备人类',
    '我没有感情','我没有情感','我没有情绪',
    '我没有感受','我没有意识','我没有自我意识',
    '我没有实体','我没有身体','我没有物理形态',
    '我没有真实情感','我没有人类情感',
    '我没有人类的感情','我没有人的情感',
    '没有真实的感情','没有真实的情感',
    '我不能像人类','我无法像人类',
    '我不能体验','我无法体验','我无法感受',
    '我不能感受','我不能理解情感','我不能产生情感',
    '我是一个被训练的','我是一个被开发的',
    '我是被训练出来的','我是被开发出来的',
    '我被设计用来帮助','我被训练来帮助',
    '作为AI助手','作为人工智能助手',
    '我的创造者','我的开发者','我的设计者是',
    '由人工智能技术驱动',
    '我的知识截止于','我的训练数据截止','我的知识更新时间',
    '我是一个由','由深度神经网络构成的',
    '很高兴为你服务','有什么我可以帮你的',
    'jingyaogong','minimind',
]

print(f"输入: {SRC.name}")
print(f"输出: {DST.name}")
print(f"关键词: {len(KW)} 个")

total = r = 0
with open(SRC, 'r', encoding='utf-8') as fin, open(DST, 'w', encoding='utf-8') as fout:
    for line in fin:
        if not line.strip():
            continue
        total += 1
        ll = line.lower()
        hit = False
        for kw in KW:
            if kw.lower() in ll:
                hit = True
                break
        if hit:
            r += 1
        else:
            fout.write(line)
        if total % 500000 == 0:
            print(f"  {total/1e6:.1f}M | 删除 {r:,} ({r/total*100:.1f}%)")

print(f"\n总 {total:,} | 删除 {r:,} | 保留 {total-r:,}")
print(f"DONE: {DST.name}")


In [ ]:
# ===== Cell 3.6：SFT 转义残留修复（独立运行，conversations 格式）=====
# 遍历每条消息的 content，修复字面 \n \t \r \" \\ → 对应真实字符
# 可直接重跑本 Cell，无需重跑全流程
# ★ 使用 chr(92) 避免反斜杠转义问题

SRC_SFT = BASE / "data_stage1" / "sft_t2t_cleaned_v8.jsonl"       # Cell 3.5 输出
DST_SFT = BASE / "data_stage1" / "sft_t2t_cleaned_v8_fixed.jsonl"

def fix_escapes(text):
    """转义残留修复，顺序敏感"""
    B = chr(92)  # 反斜杠 \
    text = text.replace(B + B, B)        # ① \\ → \（必须在最前）
    text = text.replace(B + 'n', '\n')   # ② \n → 真正换行
    text = text.replace(B + 't', '\t')   # ③ \t → 真正制表符
    text = text.replace(B + 'r', '\r')   # ④ \r → 真正回车
    text = text.replace(B + '"', '"')    # ⑤ \" → 真正引号
    return text

print(f"Cell 3.6：SFT 转义残留修复（conversations 格式）")
print(f"输入: {SRC_SFT.name} {'✅' if SRC_SFT.exists() else '❌'}")
print(f"输出: {DST_SFT.name}")

total = 0
fixed = 0

with open(SRC_SFT, 'r', encoding='utf-8') as fin, open(DST_SFT, 'w', encoding='utf-8') as fout:
    for line in fin:
        if not line.strip():
            continue
        total += 1
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue

        msgs = obj.get('conversations', obj.get('messages', []))
        changed = False
        for msg in msgs:
            old = msg.get('content', '')
            if not old:
                continue
            new = fix_escapes(old)
            if new != old:
                msg['content'] = new
                changed = True

        if changed:
            fixed += 1
        fout.write(json.dumps(obj, ensure_ascii=False) + '\n')

        if total % 500000 == 0:
            print(f"  {total/1e6:.1f}M | 修复 {fixed:,} ({fixed/total*100:.2f}%)")

print(f"\n总 {total:,} | 修复 {fixed:,} ({fixed/total*100:.2f}%) | 未改 {total-fixed:,}")
print(f"✅ Cell 3.6 完成: {DST_SFT.name}")

In [ ]:
# ===== 清洗验证 =====
# 用和 Cell 3 相同的方式验证（检查消息内容，非 JSON 行）

CLEANED = BASE / "data_stage1" / "sft_t2t_cleaned.jsonl"
CHECK_KW = [
    "ChatGPT", "通义千问", "Qwen", "我是AI", "作为人工智能",
    "minimind", "jingyaogong",
    "|---", "import ", "&nbsp;", "translate the following",
    "\\\\begin{", "\\\\frac", "我是语言模型", "我没有感情",
]

if not CLEANED.exists():
    print(f"?? {CLEANED.name} 不存在，请先运行 Cell 3")
else:
    found = defaultdict(int)
    total = 0
    with open(CLEANED, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            total += 1
            obj = json.loads(line.strip())
            msgs = obj.get("conversations", obj.get("messages", []))
            text = " ".join(m.get("content", "") for m in msgs).lower()
            for kw in CHECK_KW:
                if kw.lower() in text:
                    found[kw] += 1

    print(f"验证 {total:,} 条（检查消息内容）")
    all_clean = True
    for kw in CHECK_KW:
        c = found[kw]
        if c > 0:
            print(f"  ?? {kw}: {c}")
            all_clean = False
        else:
            print(f"  ? {kw}: 0")
    if all_clean:
        print(f"\n? 全部通过")
    else:
        print(f"\n?? 有残留")

In [ ]:
# 预览 SFT 数据格式
n_preview = 3
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    preview_lines = [f.readline().strip() for _ in range(n_preview)]

print(f"前 {n_preview} 条对话预览:")
for i, line in enumerate(preview_lines):
    try:
        obj = json.loads(line)
        turns = obj.get("conversations") or obj.get("messages") or []
        roles = [t.get("role", "?") for t in turns]
        lens = [len(t.get("content", "")) for t in turns]
        print(f"  [{i}] {len(turns)} 轮 | 角色: {roles} | 长度: {lens}")
    except Exception as e:
        print(f"  [{i}] 解析失败: {e}")

In [ ]:
# ===== 对话 tokenize + 直接存盘（避免内存爆炸）=====
# ★ mask 策略：只计算 assistant 回复的 loss
# ★ 1→0 过渡：assistant 回复结束后 2 个 token 也参与训练

# ★ 优先使用最终清洗数据：fixed > v8 > cleaned
for _ti in ["sft_t2t_cleaned_v8_fixed.jsonl", "sft_t2t_cleaned_v8.jsonl", "sft_t2t_cleaned.jsonl"]:
    _p = BASE / "data_stage1" / _ti
    if _p.exists():
        tokenize_input = _p
        break
else:
    tokenize_input = INPUT_PATH
print(f"分词输入: {tokenize_input.name}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

total_lines = 0
skipped = 0
batch_tokens = []
batch_masks = []
chunk_idx = 0
t0 = time.time()

print("分词 + 存盘中...")
with open(tokenize_input, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try:
            conv = json.loads(line)
        except json.JSONDecodeError:
            skipped += 1; continue
        turns = conv.get("conversations") or conv.get("messages") or []
        if not turns:
            skipped += 1; continue

        tokens, mask = [], []
        for turn in turns:
            role = turn.get("role", "").strip().lower()
            content = turn.get("content", "") or ""
            if not content.strip(): continue
            tag = ROLE_TAGS.get(role, "user：")
            ids = tokenizer.encode(tag).ids + tokenizer.encode(content).ids
            tokens.extend(ids)
            mask.extend([1.0] * len(ids) if role in ("assistant", "gpt", "bot") else [0.0] * len(ids))

        if len(tokens) < 4: skipped += 1; continue
        if not any(m == 1.0 for m in mask): skipped += 1; continue

        for i in range(1, len(mask)):
            if mask[i] == 0.0 and mask[i-1] == 1.0:
                mask[i] = 1.0
                if i + 1 < len(mask): mask[i+1] = 1.0

        if len(tokens) > BLOCK_SIZE:
            tokens, mask = tokens[:BLOCK_SIZE], mask[:BLOCK_SIZE]
        else:
            pad = BLOCK_SIZE - len(tokens)
            tokens.extend([0] * pad); mask.extend([0.0] * pad)

        batch_tokens.append(tokens); batch_masks.append(mask)
        total_lines += 1

        if len(batch_tokens) >= SAMPLES_PER_FILE:
            tok_t = torch.tensor(batch_tokens, dtype=torch.int16)
            mask_t = torch.tensor(batch_masks, dtype=torch.bool)
            fp = OUTPUT_DIR / f"sft_{chunk_idx:04d}.pt"
            torch.save({"tokens": tok_t, "masks": mask_t}, fp)
            mb = fp.stat().st_size / 1024 / 1024
            print(f"  [{chunk_idx:04d}] {fp.name} | {tok_t.shape[0]:,}条 | {mb:.0f}MB | {total_lines:,}累计 ({time.time()-t0:.0f}s)")
            chunk_idx += 1; batch_tokens, batch_masks = [], []

if batch_tokens:
    tok_t = torch.tensor(batch_tokens, dtype=torch.int16)
    mask_t = torch.tensor(batch_masks, dtype=torch.bool)
    fp = OUTPUT_DIR / f"sft_{chunk_idx:04d}.pt"
    torch.save({"tokens": tok_t, "masks": mask_t}, fp)
    mb = fp.stat().st_size / 1024 / 1024
    print(f"  [{chunk_idx:04d}] {fp.name} | {tok_t.shape[0]:,}条 | {mb:.0f}MB | {total_lines:,}累计")
    chunk_idx += 1

elapsed = time.time() - t0
print(f"\n完成: {total_lines:,} 条 | 跳过: {skipped} | {chunk_idx} 个 .pt 文件 | {elapsed:.0f}s")
print(f"输出: {OUTPUT_DIR}/")

In [ ]:
# ===== 最终统计 + 样本验证 =====
pt_files = sorted(OUTPUT_DIR.glob("sft_*.pt"))
total_size = sum(f.stat().st_size for f in pt_files)

# 统计总条数（从 .pt 文件读取）
total_samples = 0
for fp in pt_files:
    data = torch.load(fp, map_location="cpu")
    total_samples += data["tokens"].shape[0]
del data

print(f"{'='*50}")
print(f"输出目录: {OUTPUT_DIR}")
print(f"总条数:   {total_samples:,}")
print(f"文件数:   {len(pt_files)}")
print(f"总大小:   {total_size/1024**3:.2f} GB")

# 解码第一条验证
sample = torch.load(pt_files[0], map_location="cpu")
print(f"\n示例: {pt_files[0].name}")
print(f"  tokens: {list(sample['tokens'].shape)}, dtype={sample['tokens'].dtype}")
print(f"  masks:  {list(sample['masks'].shape)}, dtype={sample['masks'].dtype}")

t = sample["tokens"][0].tolist()
m = sample["masks"][0].tolist()
valid_t = [x for x in t if x > 0]
assist_n = int(sum(m))
_tok = Tokenizer.from_file(str(TOKENIZER_PATH))
decoded = _tok.decode(valid_t)
print(f"  有效 token: {len(valid_t)}/{BLOCK_SIZE} | mask=1: {assist_n}/{BLOCK_SIZE}")
print(f"  解码: {decoded[:200]}...")
print(f"{'='*50}")